# Classical Fine-Tuning with Raw PyTorch and Lightning
## Human Activity Recognition — Sensor Domain Adaptation

Fine-tuning is not an LLM-specific technique. Any pre-trained neural network
can be fine-tuned on new data from a shifted distribution.

**This notebook demonstrates the concept using accelerometer sensor data.**
A model pre-trained on data from subjects 1–16 degrades when deployed on
new subjects 17–22 whose movement patterns differ. Fine-tuning on a small
amount of new-subject data cheaply restores performance.

| Section | Framework | What it shows |
|---------|-----------|---------------|
| **A** | Raw PyTorch | Manual training loop, explicit freeze/unfreeze |
| **B** | PyTorch Lightning | Same strategies, boilerplate eliminated |

**Dataset:** UCI HAR — 30 subjects, 6 activity classes, 561 features  
**Model:** ~150K-param MLP — trains in seconds on CPU, <1 min on GPU  

| | LLM fine-tuning | This notebook |
|-|----------------|---------------|
| Model size | 1B–70B | ~150K params |
| Adapter | LoRA / QLoRA | Frozen head / full unfreeze |
| Framework | TRL / Unsloth | Raw PyTorch / Lightning |
| Training time | Hours–days | Seconds–minutes |

## Environment Setup

```bash
cd projects/huggingface_trl
uv sync --no-install-workspace
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=huggingface_trl
```

Select the **`huggingface_trl`** kernel in JupyterLab.

In [ ]:
import sys, pathlib, urllib.request, zipfile, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

print(f'Python    {sys.version}')
print(f'PyTorch   {torch.__version__}')
print(f'Lightning {L.__version__}')
print(f'CUDA      {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device    {DEVICE}')

## Download & Load UCI HAR Dataset

> **Gotcha #1 — The dataset files use multiple spaces as delimiters.**
> The `.txt` files are fixed-width space-separated. Using `sep=' '` in
> `pd.read_csv` fails because multiple spaces become empty columns.
> Use `sep=r'\s+'` (regex) or `np.loadtxt` instead.

In [ ]:
DATA_DIR = pathlib.Path('/tmp/uci_har')
ZIP_PATH = DATA_DIR / 'har.zip'
URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'

if not (DATA_DIR / 'UCI HAR Dataset').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading UCI HAR (~60 MB)...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR)
    print('Done.')
else:
    print('Already downloaded.')

HAR = DATA_DIR / 'UCI HAR Dataset'

In [ ]:
def load_split(split: str):
    """Load features, labels (0-indexed), and subject IDs for train or test."""
    X = pd.read_csv(HAR / split / f'X_{split}.txt', sep=r'\s+', header=None).values.astype(np.float32)
    y = pd.read_csv(HAR / split / f'y_{split}.txt', sep=r'\s+', header=None).values.ravel().astype(np.int64) - 1
    subjects = pd.read_csv(HAR / split / f'subject_{split}.txt', sep=r'\s+', header=None).values.ravel()
    return X, y, subjects

X_train, y_train, subj_train = load_split('train')
X_test,  y_test,  subj_test  = load_split('test')

ACTIVITY_NAMES = ['Walking', 'Walking Upstairs', 'Walking Downstairs',
                  'Sitting', 'Standing', 'Laying']

print(f'Train: {X_train.shape}  unique subjects: {sorted(set(subj_train))}')
print(f'Test:  {X_test.shape}   unique subjects: {sorted(set(subj_test))}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
axes[0].bar([ACTIVITY_NAMES[i] for i in unique], counts, color='steelblue')
axes[0].set_title('Samples per Class (train)')
axes[0].tick_params(axis='x', rotation=30)

# Subject distribution
s_unique, s_counts = np.unique(subj_train, return_counts=True)
axes[1].bar(s_unique, s_counts, color='darkorange')
axes[1].set_title('Samples per Subject (train)')
axes[1].set_xlabel('Subject ID')

plt.tight_layout()
plt.show()

## Subject Split Strategy

We split the **training** subjects into two groups:

| Group | Subjects | Samples | Purpose |
|-------|----------|---------|--------|
| Pre-train | 1, 3, 5, 6, 7, 8, 11, 14, 15, 16 | ~3,300 | Learn general features |
| Fine-tune | 17, 19, 21, 22 | ~1,300 | Simulate new users |
| **Hold-out** | 2, 4, 9, 10, 12, 13, 18, 20, 24 | ~2,947 | Final evaluation only |

The hold-out set is the **official UCI test split** — never touched until
the final evaluation, regardless of which strategy we use.

In [ ]:
PRETRAIN_SUBJ = {1, 3, 5, 6, 7, 8, 11, 14, 15, 16}
FINETUNE_SUBJ = {17, 19, 21, 22}

pt_mask = np.isin(subj_train, list(PRETRAIN_SUBJ))
ft_mask = np.isin(subj_train, list(FINETUNE_SUBJ))

X_pt, y_pt = X_train[pt_mask], y_train[pt_mask]
X_ft, y_ft = X_train[ft_mask], y_train[ft_mask]

print(f'Pre-train samples : {len(X_pt)}')
print(f'Fine-tune samples : {len(X_ft)}')
print(f'Hold-out (test)   : {len(X_test)}')

In [ ]:
def make_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

loader_pt   = make_loader(X_pt,   y_pt)    # pre-train
loader_ft   = make_loader(X_ft,   y_ft)    # fine-tune
loader_test = make_loader(X_test, y_test, shuffle=False)  # hold-out

print(f'Batches — pre-train: {len(loader_pt)}  fine-tune: {len(loader_ft)}  test: {len(loader_test)}')

## Model: `ActivityMLP`

A two-layer MLP with an explicit **encoder** (learns general sensor features)
and a **head** (maps features to class logits). Keeping them as separate
`nn.Sequential` blocks makes freeze/unfreeze trivial.

In [ ]:
class ActivityMLP(nn.Module):
    def __init__(self, in_features: int = 561, hidden: int = 256, num_classes: int = 6):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_features, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 128),          nn.ReLU(), nn.Dropout(0.3),
        )
        self.head = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))


sample_model = ActivityMLP()
total  = sum(p.numel() for p in sample_model.parameters())
print(f'Total parameters: {total:,}')
del sample_model

---
## Section A — Raw PyTorch

A minimal training loop: forward pass, cross-entropy loss, Adam update.
No framework magic — every step is visible.

### Training and Evaluation Helpers

> **Gotcha #2 — Always call `model.train()` / `model.eval()`.**
> Forgetting `model.eval()` before evaluation leaves **Dropout active**,
> so each forward pass randomly zeroes activations. Eval accuracy will be
> noisy and artificially low — a confusing bug that is easy to miss.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    """One epoch of train (optimizer given) or eval (optimizer=None)."""
    if optimizer:
        model.train()   # enables dropout
    else:
        model.eval()    # disables dropout — DO NOT forget this

    total_loss, correct, n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()

    ctx = torch.no_grad() if optimizer is None else torch.enable_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            if optimizer:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y_batch)
            correct    += (logits.argmax(1) == y_batch).sum().item()
            n          += len(y_batch)

    return total_loss / n, correct / n


def evaluate(model, loader):
    """Return accuracy and all predictions for confusion matrix."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds = model(X_batch.to(DEVICE)).argmax(1).cpu()
            all_preds.append(preds)
            all_labels.append(y_batch)
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return accuracy_score(labels, preds), preds, labels


def plot_confusion(preds, labels, title=''):
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=[a[:8] for a in ACTIVITY_NAMES])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

### Pre-Training on Subjects 1–16

In [ ]:
PRETRAIN_EPOCHS = 15
LR_PRETRAIN     = 1e-3

pretrained_model = ActivityMLP().to(DEVICE)
optimizer_pt = torch.optim.Adam(pretrained_model.parameters(), lr=LR_PRETRAIN)

train_losses, train_accs = [], []
for epoch in range(1, PRETRAIN_EPOCHS + 1):
    loss, acc = run_epoch(pretrained_model, loader_pt, optimizer_pt)
    train_losses.append(loss)
    train_accs.append(acc)
    if epoch % 5 == 0:
        print(f'Epoch {epoch:2d} | loss {loss:.4f} | train acc {acc:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(train_losses); axes[0].set_title('Pre-train Loss')
axes[1].plot(train_accs);   axes[1].set_title('Pre-train Accuracy')
plt.tight_layout(); plt.show()

# Save weights for reuse later
pretrained_weights = copy.deepcopy(pretrained_model.state_dict())

In [ ]:
acc_test_pretrained, preds, labels = evaluate(pretrained_model, loader_test)
print(f'Hold-out accuracy (pre-trained, no adaptation): {acc_test_pretrained:.3f}')
plot_confusion(preds, labels, 'Pre-trained → Hold-out test')

### Domain Shift: New Subjects Without Adaptation

> **Gotcha #3 — High training accuracy does not guarantee domain transfer.**
> The model achieves ~90%+ on the hold-out test set because those subjects
> share the same distribution as the training subjects (UCI's official split).
> Our fine-tune subjects were deliberately excluded from pre-training — they
> represent a **new deployment domain** and performance will be lower.

In [ ]:
acc_ft_domain, preds_ft, labels_ft = evaluate(pretrained_model, loader_ft)
print(f'Fine-tune subjects accuracy WITHOUT adaptation: {acc_ft_domain:.3f}')
print(f'(Compare to hold-out accuracy: {acc_test_pretrained:.3f})')
plot_confusion(preds_ft, labels_ft, 'Pre-trained → New subjects (no adaptation)')

### Strategy 1 — Frozen Encoder Fine-Tuning

Freeze the encoder, train only the classification head.
Best when: the new dataset is small, you want fast adaptation, and you
trust that the encoder's features are general enough.

> **Gotcha #4 — Pass only trainable parameters to the optimizer.**
> Setting `param.requires_grad = False` stops gradient computation, but
> if you pass `model.parameters()` to the optimizer, the frozen parameters
> are still tracked in the optimizer state (momentum buffers, etc.).
> This wastes memory and can cause subtle update errors.
> Always filter: `filter(lambda p: p.requires_grad, model.parameters())`.

In [ ]:
FT_EPOCHS = 10

model_frozen = ActivityMLP().to(DEVICE)
model_frozen.load_state_dict(copy.deepcopy(pretrained_weights))

# Freeze encoder
for param in model_frozen.encoder.parameters():
    param.requires_grad = False

frozen_params    = sum(p.numel() for p in model_frozen.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
print(f'Frozen: {frozen_params:,}  Trainable: {trainable_params:,}')

# Only pass trainable params to optimizer — avoids wasted optimizer state
optimizer_frozen = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_frozen.parameters()),
    lr=1e-3,
)

frozen_accs = []
for epoch in range(1, FT_EPOCHS + 1):
    _, acc_train = run_epoch(model_frozen, loader_ft, optimizer_frozen)
    _, acc_val, _ = evaluate(model_frozen, loader_ft)[0], *evaluate(model_frozen, loader_ft)[1:]
    acc_val_num = evaluate(model_frozen, loader_ft)[0]
    frozen_accs.append(acc_val_num)

acc_frozen_ft = evaluate(model_frozen, loader_ft)[0]
acc_frozen_test = evaluate(model_frozen, loader_test)[0]
print(f'Frozen FT — fine-tune acc: {acc_frozen_ft:.3f}  hold-out acc: {acc_frozen_test:.3f}')

### Strategy 2 — Full Fine-Tuning (Smaller LR)

Unfreeze all layers and train with a **lower learning rate** than pre-training.

> **Gotcha #5 — Using the pre-training LR causes catastrophic forgetting.**
> At LR=1e-3, the model overwrites the useful encoder features learned
> during pre-training within the first few batches. The result is often
> *worse* than frozen fine-tuning on a small dataset.
> Use LR ≈ 1/10th of the pre-training LR (here: 1e-4).

In [ ]:
model_full = ActivityMLP().to(DEVICE)
model_full.load_state_dict(copy.deepcopy(pretrained_weights))

# All parameters trainable — note smaller LR
optimizer_full = torch.optim.Adam(model_full.parameters(), lr=1e-4)

full_accs = []
for epoch in range(1, FT_EPOCHS + 1):
    run_epoch(model_full, loader_ft, optimizer_full)
    full_accs.append(evaluate(model_full, loader_ft)[0])

acc_full_ft   = evaluate(model_full, loader_ft)[0]
acc_full_test = evaluate(model_full, loader_test)[0]
print(f'Full FT   — fine-tune acc: {acc_full_ft:.3f}  hold-out acc: {acc_full_test:.3f}')

# Plot frozen vs full fine-tuning curves
plt.figure(figsize=(7, 3))
plt.plot(frozen_accs, label='Frozen encoder (LR=1e-3)')
plt.plot(full_accs,   label='Full fine-tune (LR=1e-4)')
plt.axhline(acc_ft_domain, color='grey', linestyle='--', label='No adaptation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (fine-tune subjects)')
plt.title('Fine-Tuning Strategies'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
lr_results = {}
for lr in [1e-3, 1e-4, 1e-5]:
    m = ActivityMLP().to(DEVICE)
    m.load_state_dict(copy.deepcopy(pretrained_weights))
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(FT_EPOCHS):
        run_epoch(m, loader_ft, opt)
    lr_results[lr] = evaluate(m, loader_ft)[0]
    print(f'LR {lr:.0e} → fine-tune acc: {lr_results[lr]:.3f}')

plt.figure(figsize=(5, 3))
plt.bar([f'{lr:.0e}' for lr in lr_results], list(lr_results.values()), color='teal')
plt.title('Full Fine-Tuning: LR Sensitivity')
plt.xlabel('Learning Rate'); plt.ylabel('Accuracy'); plt.tight_layout(); plt.show()

In [ ]:
# Train from scratch on fine-tune subjects only — for comparison
model_scratch = ActivityMLP().to(DEVICE)
opt_scratch = torch.optim.Adam(model_scratch.parameters(), lr=1e-3)
for _ in range(FT_EPOCHS):
    run_epoch(model_scratch, loader_ft, opt_scratch)
acc_scratch_ft   = evaluate(model_scratch, loader_ft)[0]
acc_scratch_test = evaluate(model_scratch, loader_test)[0]

print('\n=== Results Summary ===')
rows = [
    ('Pre-trained, no adaptation', acc_ft_domain,    acc_test_pretrained),
    ('Frozen encoder FT (LR=1e-3)', acc_frozen_ft,  acc_frozen_test),
    ('Full fine-tuning (LR=1e-4)',  acc_full_ft,     acc_full_test),
    ('Train from scratch',          acc_scratch_ft,  acc_scratch_test),
]
print(f'{"Strategy":<32} {"FT-subj acc":>12} {"Hold-out acc":>13}')
print('-' * 60)
for name, ft_acc, test_acc in rows:
    print(f'{name:<32} {ft_acc:>12.3f} {test_acc:>13.3f}')

---
## Section B — PyTorch Lightning

Lightning wraps the training loop in a `LightningModule`, eliminating
boilerplate while keeping the same fine-tuning strategies.
Benefits over raw PyTorch:
- Automatic `train()`/`eval()` mode switching
- Built-in logging, early stopping, checkpointing
- Device management handled automatically
- Same code runs on CPU, GPU, or multi-GPU

In [ ]:
class LitActivityMLP(L.LightningModule):
    def __init__(self, lr: float = 1e-3, freeze_encoder: bool = False):
        super().__init__()
        self.save_hyperparameters()
        self.model = ActivityMLP()
        self.criterion = nn.CrossEntropyLoss()

        if freeze_encoder:
            for p in self.model.encoder.parameters():
                p.requires_grad = False

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc',  acc,  prog_bar=True)
        return loss

    def training_step(self, batch, _):   return self._step(batch, 'train')
    def validation_step(self, batch, _): return self._step(batch, 'val')

    def configure_optimizers(self):
        params = filter(lambda p: p.requires_grad, self.model.parameters())
        return torch.optim.Adam(params, lr=self.hparams.lr)


print('LitActivityMLP defined.')

### Pre-Training with Lightning


In [ ]:
lit_pretrain = LitActivityMLP(lr=1e-3, freeze_encoder=False)

ckpt_callback = ModelCheckpoint(
    dirpath='./har_checkpoints', filename='pretrained-{epoch:02d}',
    monitor='val_acc', mode='max', save_top_k=1,
)

trainer_pre = L.Trainer(
    max_epochs=15,
    callbacks=[ckpt_callback],
    enable_progress_bar=True,
    log_every_n_steps=5,
    logger=False,
)
trainer_pre.fit(lit_pretrain, loader_pt, loader_ft)
best_ckpt = ckpt_callback.best_model_path
print(f'Best checkpoint: {best_ckpt}')

### Fine-Tuning with Lightning

> **Gotcha #6 — Use `load_from_checkpoint`, not `torch.load`.**
> Lightning checkpoints store hyperparameters and model state together.
> Using `torch.load()` returns a raw dict — you must manually re-build
> the `LightningModule` and call `load_state_dict()` yourself, which is
> error-prone. `LitModel.load_from_checkpoint(path)` does it correctly
> in one call, restoring hparams and weights together.

In [ ]:
# Correct: load_from_checkpoint restores hparams + weights
lit_ft = LitActivityMLP.load_from_checkpoint(
    best_ckpt,
    lr=1e-4,             # override LR for fine-tuning
    freeze_encoder=True, # frozen encoder strategy
)

trainer_ft = L.Trainer(
    max_epochs=10,
    callbacks=[EarlyStopping(monitor='val_acc', patience=3, mode='max')],
    enable_progress_bar=True,
    log_every_n_steps=5,
    logger=False,
)
trainer_ft.fit(lit_ft, loader_ft, loader_ft)

results = trainer_ft.validate(lit_ft, loader_test, verbose=False)
print(f'Lightning frozen FT — hold-out acc: {results[0]["val_acc"]:.3f}')

## Summary

### Gotcha Recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | Space-separated files use multiple spaces | `pd.read_csv(sep=r'\s+')` not `sep=' '` |
| 2 | Eval metrics are noisy | Always call `model.eval()` before evaluation |
| 3 | Good train accuracy ≠ good domain transfer | Evaluate on held-out domain before claiming success |
| 4 | Frozen params in optimizer state | Pass `filter(lambda p: p.requires_grad, ...)` to optimizer |
| 5 | Same LR → catastrophic forgetting | Use ~1/10th of pre-training LR for full fine-tuning |
| 6 | `torch.load` for Lightning checkpoints | Use `LitModel.load_from_checkpoint(path)` |

### When to Use Each Strategy

| Strategy | Use when |
|----------|----------|
| Frozen encoder | Target dataset is small (<500 samples), fast iteration needed |
| Full fine-tuning | Target dataset is larger, best accuracy required, use small LR |
| Train from scratch | Target domain is completely different from pre-training domain |

**Next steps:** replace the MLP encoder with a 1D CNN on the raw inertial
signals (in `UCI HAR Dataset/train/Inertial Signals/`), add `lr_scheduler`
cosine decay, or try multi-task pre-training (all 6 activities at once then
fine-tune on a person-specific 3-class subset).